<a href="https://colab.research.google.com/github/BuiltbyMkulimah/Predictive-Drought-Severity-using-Satelite-Indices/blob/main/PREDICTING_DROUGHT_SEVERITY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import os
import zipfile

def extraction(target_url,destination):
  if not os.path.exists(destination):
    os.makedirs(destination)


  file_name=target_url.split('/')[-1] + '.zip'
  savepath=os.path.join(destination,file_name)

  if not os.path.exists(savepath):
    r=requests.get(target_url)
    with open(savepath,'wb') as f:
      f.write(r.content)
    with zipfile.ZipFile(savepath,'r') as z:
      z.extractall(destination)
  else:
    print('already downloaded')

datasets=[
    {
    'url':'https://data.humdata.org/dataset/64f10c44-56e3-45a2-81cd-d49bb409890b/resource/04b179e0-2efd-4e3d-b6d8-6a7635e935ef/download/93926f75-8d59-4809-9665-8af1af746097',
    'folder':'./data/precipitation'

    },
    {
    'url':'https://edcintl.cr.usgs.gov/downloads/sciweb1/shared/fews/web/africa/east/pentadal/chirps/seasaccum/octdec/anom/lta/downloads/pentadal/ea_chirps_seasaccum_anom_octdec_202372_lta.zip',
    'folder':'./data/rainfall'
    },
    {
      'url':'https://data.humdata.org/dataset/adcafb60-7ec8-480a-96ec-9a81a027900c/resource/f6386de5-67d4-4e0b-a081-868a7d7467d2/download/554f5519-6f09-42c8-8d95-a27fba64a608',
      'folder':'./data/ndvi'
    }


          ]
if __name__=='__main__':
  for item in datasets:
    try:
      extraction(item['url'],item['folder'])
    except Exception as e:
      print(f"Error in processing {item['url']}: {e}")



In [ ]:
import rasterio
from rasterio.mask import mask
import pandas as pd
import numpy as np
import os

for roof,dir,files in os.walk('data'):
  for file in files:
    print(os.path.join(roof,file))

def transformation(filename):
  print(f"\nProcessing file: {filename}")
  with rasterio.open(filename) as src:
    data=src.read (1)
    print(f"Raw data shape: {data.shape}")
    print(f"Raw data min: {np.nanmin(data)} max: {np.nanmax(data)} (before nodata handling)")

    valid_pixels_count = np.count_nonzero(data != src.nodata)
    print(f"Number of valid pixels (not nodata): {valid_pixels_count}")

    data=np.where(data ==src.nodata,np.nan,data)
    nan_count = np.count_nonzero(np.isnan(data))
    print(f"Number of NaN values after nodata conversion: {nan_count}")
    if np.all(np.isnan(data)) or (np.nanmax(data) - np.nanmin(data) == 0):
      print(f"Warning: Data for {filename} is either all NaN or has zero range. Returning array of zeros/NaNs.")
      if np.nanmax(data) - np.nanmin(data) == 0 and not np.isnan(np.nanmin(data)):
        return np.zeros_like(data.flatten())
      else:
        return np.full_like(data.flatten(), np.nan)

    normalised_data=(data - np.nanmin(data)) / (np.nanmax(data) - np.nanmin(data))

    print(f"Normalised data min: {np.nanmin(normalised_data)} max: {np.nanmax(normalised_data)}")
    return normalised_data.flatten()

rainfall_features = transformation('./data/ndvi/fapar_anomaly_0101_2026.tif')
ndvi_features = transformation('./data/precipitation/spi_0206_2026.tif')

print(f"Rainfall array length: {len(rainfall_features)}")
print(f"NDVI array length: {len(ndvi_features)}")



data/ndvi/554f5519-6f09-42c8-8d95-a27fba64a608.zip
data/ndvi/fapar_anomaly_0101_2026.tif
data/ndvi/fapar_anomaly_0111_2026.tif
data/ndvi/fapar_anomaly_0121_2026.tif
data/precipitation/spi_0248_2026.tif
data/precipitation/spi_0224_2026.tif
data/precipitation/spi_0206_2026.tif
data/precipitation/spi_0203_2026.tif
data/precipitation/spi_0112_2026.tif
data/precipitation/spi_0209_2026.tif
data/precipitation/spi_0106_2026.tif
data/precipitation/spi_0212_2026.tif
data/precipitation/spi_0103_2026.tif
data/precipitation/spi_0101_2026.tif
data/precipitation/spi_0124_2026.tif
data/precipitation/spi_0109_2026.tif
data/precipitation/spi_0148_2026.tif
data/precipitation/93926f75-8d59-4809-9665-8af1af746097.zip
data/rainfall/ea_chirps_seasaccum_anom_octdec_202372_lta.tif
data/rainfall/ea_chirps_seasaccum_anom_octdec_202372_lta.zip.zip

Processing file: ./data/ndvi/fapar_anomaly_0101_2026.tif
Raw data shape: (2168, 4337)
Raw data min: -9999 max: 12 (before nodata handling)
Number of valid pixels (not 

data/ndvi/554f5519-6f09-42c8-8d95-a27fba64a608.zip
data/ndvi/fapar_anomaly_0101_2026.tif
data/ndvi/fapar_anomaly_0111_2026.tif
data/ndvi/fapar_anomaly_0121_2026.tif
data/precipitation/spi_0248_2026.tif
data/precipitation/spi_0224_2026.tif
data/precipitation/spi_0206_2026.tif
data/precipitation/spi_0203_2026.tif
data/precipitation/spi_0112_2026.tif
data/precipitation/spi_0209_2026.tif
data/precipitation/spi_0106_2026.tif
data/precipitation/spi_0212_2026.tif
data/precipitation/spi_0103_2026.tif
data/precipitation/spi_0101_2026.tif
data/precipitation/spi_0124_2026.tif
data/precipitation/spi_0109_2026.tif
data/precipitation/spi_0148_2026.tif
data/precipitation/93926f75-8d59-4809-9665-8af1af746097.zip
data/rainfall/ea_chirps_seasaccum_anom_octdec_202372_lta.tif
data/rainfall/ea_chirps_seasaccum_anom_octdec_202372_lta.zip.zip

Processing file: ./data/ndvi/fapar_anomaly_0101_2026.tif
Raw data shape: (2168, 4337)
Raw data min: -9999 max: 12 (before nodata handling)
Number of valid pixels (not nodata): 235941
Number of NaN values after nodata conversion: 9166675
Normalised data min: 0.0 max: 1.0

Processing file: ./data/precipitation/spi_0206_2026.tif
Raw data shape: (180, 360)
Raw data min: -9999 max: 4 (before nodata handling)
Number of valid pixels (not nodata): 3498
Number of NaN values after nodata conversion: 61302
Normalised data min: 0.0 max: 1.0
Rainfall array length: 9402616
NDVI array length: 64800

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform,transform,reproject, Resampling
import numpy as np

def arrange_the_rasters(source_path,template_file,outputpath):
  with rasterio.open(template_file) as template:
    dst_crs=template.crs
    dst_height=template.height
    dst_width=template.width
    dst_transform=template.transform
    dst_nodata_val = template.nodata

  with rasterio.open(source_path) as src:
    destination = np.full((dst_height,dst_width), dst_nodata_val if dst_nodata_val is not None else np.nan, dtype=np.float32)

    _src_nodata = src.nodata if src.nodata is not None else -9999
    _dst_nodata = dst_nodata_val if dst_nodata_val is not None else _src_nodata

    reproject(
      source=rasterio.band(src,1),
      destination=destination,
      src_transform=src.transform,
      src_crs=src.crs,
      src_nodata=_src_nodata,
      dst_transform=dst_transform,
      dst_crs=dst_crs,
      dst_nodata=_dst_nodata,
      resampling=Resampling.bilinear
    )
  with rasterio.open(
       outputpath, 'w',
       driver='GTiff',
       height=dst_height,
       width=dst_width,
       count=1,
       dtype=destination.dtype,
       crs=dst_crs,
       transform=dst_transform,
       nodata=dst_nodata_val
       ) as dst:
            dst.write(destination, 1)

arrange_the_rasters('data/ndvi/fapar_anomaly_0101_2026.tif','data/precipitation/spi_0206_2026.tif','ndvi_low_res.tif')



In [ ]:
import rasterio
import numpy as np

def extract_aligned_data(rainfall_path, ndvi_path):
    with rasterio.open(rainfall_path) as src1, rasterio.open(ndvi_path) as src2:
        rain = src1.read(1)
        ndvi = src2.read(1)
        mask = (rain != -9999) & (ndvi != -9999)

        valid_rain = rain[mask]
        valid_ndvi = ndvi[mask]
        rain_norm = (valid_rain - valid_rain.min()) / (valid_rain.max() - valid_rain.min())
        ndvi_norm = (valid_ndvi - valid_ndvi.min()) / (valid_ndvi.max() - valid_ndvi.min())

        return pd.DataFrame({'rainfall': rain_norm, 'ndvi': ndvi_norm})


df_clean = extract_aligned_data('data/precipitation/spi_0206_2026.tif', 'ndvi_aligned.tif')

print(f"Data remaining: {len(df_clean)} rows")
print(df_clean.head())

Data remaining: 1613 rows
   rainfall      ndvi
0  0.666667  0.301563
1  0.666667  0.296735
2  0.555556  0.296352
3  0.666667  0.312665
4  0.666667  0.255059


Data remaining: 1613 rows
   rainfall      ndvi
0  0.666667  0.301563
1  0.666667  0.296735
2  0.555556  0.296352
3  0.666667  0.312665
4  0.666667  0.255059

In [ ]:

df_clean['month'] = 6

df_clean['rainfall_lag1'] = df_clean['rainfall'].shift(1)
df_clean['rainfall_rolling_mean'] = df_clean['rainfall'].rolling(window=3).mean()
df_clean = df_clean.dropna()
X = df_clean[['rainfall', 'ndvi', 'month', 'rainfall_lag1', 'rainfall_rolling_mean']]

In [ ]:

def classify_drought(spi_value):
    if spi_value < -1.5: return 2
    if spi_value < -1.0: return 1
    return 0

df_clean['drought_class'] = df_clean['rainfall'].apply(classify_drought)

/tmp/ipykernel_5885/2558975862.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['drought_class'] = df_clean['rainfall'].apply(classify_drought)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = df_clean[['rainfall', 'ndvi', 'month']]
y = df_clean['drought_class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       322

    accuracy                           1.00       322
   macro avg       1.00      1.00      1.00       322
weighted avg       1.00      1.00      1.00       322



 precision    recall  f1-score   support

           0       1.00      1.00      1.00       322

    accuracy                           1.00       322
   macro avg       1.00      1.00      1.00       322
weighted avg       1.00      1.00      1.00       322  precision    recall  f1-score   support

           0       1.00      1.00      1.00       322

    accuracy                           1.00       322
   macro avg       1.00      1.00      1.00       322
weighted avg       1.00      1.00      1.00       322

In [ ]:
import matplotlib.pyplot as plt
importances = model.feature_importances_
print(dict(zip(X.columns, importances)))

{'rainfall': np.float64(0.0), 'ndvi': np.float64(0.0), 'month': np.float64(0.0)}


In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, X, y, cv=5)
print(f"Average accuracy across 5 folds: {scores.mean()}")

Average accuracy across 5 folds: 1.0
